<a href="https://colab.research.google.com/github/couldbecarissa/Credit-Score-Evaluation/blob/main/benford_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Benford's Law Analysis — Lending Club Loan Dataset

This notebook applies five statistical tests to detect whether financial columns in the Lending Club dataset conform to Benford's Law.

**Columns tested:** `loan_amnt`, `annual_inc`, `funded_amnt`

**Tests:**
| Test | What It Measures | Null Hypothesis H₀ |
|---|---|---|
| Chi-Squared | Aggregate digit frequency | Observed ≈ Benford expected |
| Kolmogorov-Smirnov | Cumulative distribution gap | Mantissa CDF ≈ Uniform |
| Mean Absolute Deviation | Average per-digit deviation | MAD < conformity threshold |
| Z-Score (per digit) | Individual digit anomaly | Each digit proportion ≈ Benford |
| Mantissa Arc Test | Shape of mantissa distribution | Mantissa mean vector ≈ (0, 0) |

## 1. Imports & Constants

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
import os

warnings.filterwarnings("ignore")
%matplotlib inline

# Benford's Law expected probabilities: P(d) = log10(1 + 1/d)
DIGITS = np.arange(1, 10)
BENFORD_PROBS = np.log10(1 + 1 / DIGITS)

print("Benford expected probabilities:")
for d, p in zip(DIGITS, BENFORD_PROBS):
    print(f"  P(leading digit = {d}) = {p:.4f}")

Benford expected probabilities:
  P(leading digit = 1) = 0.3010
  P(leading digit = 2) = 0.1761
  P(leading digit = 3) = 0.1249
  P(leading digit = 4) = 0.0969
  P(leading digit = 5) = 0.0792
  P(leading digit = 6) = 0.0669
  P(leading digit = 7) = 0.0580
  P(leading digit = 8) = 0.0512
  P(leading digit = 9) = 0.0458


## 2. Helper Functions

In [2]:
def extract_leading_digits(series: pd.Series) -> pd.Series:
    s = series.dropna()
    s = s[s > 0].astype(float)
    magnitudes = np.floor(np.log10(s))
    normalized = s / (10 ** magnitudes)
    return np.floor(normalized).astype(int)


def extract_mantissas(series: pd.Series) -> np.ndarray:
    """
    Mantissa m = log10(x) - floor(log10(x)), so m in [0, 1).
    For Benford-conforming data, m should be uniform on [0, 1).
    """
    s = series.dropna()
    s = s[s > 0].astype(float)
    log_vals = np.log10(s)
    return log_vals - np.floor(log_vals)


def observed_frequencies(leading_digits: pd.Series) -> np.ndarray:
    return np.array([(leading_digits == d).sum() for d in DIGITS])

## 3. Statistical Tests

### Test 1: Chi-Squared Goodness-of-Fit

$$\chi^2 = \sum_{d=1}^{9} \frac{(O_d - E_d)^2}{E_d}$$

- **H₀:** Observed digit frequencies match Benford expected frequencies  
- **Reject H₀** if $\chi^2 > \chi^2_{0.05,\, df=8} = 15.507$

In [3]:
def chi_squared_test(leading_digits: pd.Series) -> dict:
    n = len(leading_digits)
    observed = observed_frequencies(leading_digits)
    expected = BENFORD_PROBS * n

    chi2_stat = np.sum((observed - expected) ** 2 / expected)
    p_value = 1 - stats.chi2.cdf(chi2_stat, df=8)
    critical = stats.chi2.ppf(0.95, df=8)

    return {
        "test": "Chi-Squared",
        "statistic": chi2_stat,
        "p_value": p_value,
        "critical_value": critical,
        "reject_H0": chi2_stat > critical,
        "observed": observed,
        "expected": expected,
        "n": n,
    }

### Test 2: Kolmogorov-Smirnov Test

$$D = \max_{m \in [0,1)} |F_{\text{observed}}(m) - F_{\text{Benford}}(m)|$$

- Tests the mantissa distribution directly against the uniform CDF (since Benford mantissas are uniform on [0,1))  
- **H₀:** Mantissa CDF ≈ Uniform  
- **Reject H₀** if $p < 0.05$

In [4]:
def ks_test(series: pd.Series) -> dict:
    mantissas = extract_mantissas(series)
    ks_stat, p_value = stats.kstest(mantissas, "uniform")

    return {
        "test": "Kolmogorov-Smirnov",
        "statistic": ks_stat,
        "p_value": p_value,
        "critical_value": 1.36 / np.sqrt(len(mantissas)),  # alpha=0.05
        "reject_H0": p_value < 0.05,
        "n": len(mantissas),
    }

### Test 3: Mean Absolute Deviation (MAD)

$$\text{MAD} = \frac{1}{9} \sum_{d=1}^{9} |p_{\text{obs}}(d) - p_{\text{Benford}}(d)|$$

Nigrini (2012) conformity thresholds:

| MAD range | Interpretation |
|---|---|
| < 0.006 | Close conformity |
| 0.006 – 0.012 | Acceptable conformity |
| 0.012 – 0.015 | Marginally acceptable |
| ≥ 0.015 | Nonconformity |

In [5]:
def mad_test(leading_digits: pd.Series) -> dict:
    n = len(leading_digits)
    observed_props = observed_frequencies(leading_digits) / n
    mad = np.mean(np.abs(observed_props - BENFORD_PROBS))

    if mad < 0.006:
        conformity = "Close conformity"
    elif mad < 0.012:
        conformity = "Acceptable conformity"
    elif mad < 0.015:
        conformity = "Marginally acceptable"
    else:
        conformity = "Nonconformity"

    return {
        "test": "Mean Absolute Deviation",
        "statistic": mad,
        "conformity": conformity,
        "reject_H0": mad >= 0.015,
        "n": n,
    }

### Test 4: Z-Score Per Digit

$$Z_d = \frac{|p_{\text{obs}}(d) - p_{\text{expected}}(d)| - \frac{1}{2n}}{\sqrt{\frac{p_{\text{expected}}(d)(1 - p_{\text{expected}}(d))}{n}}}$$

- **H₀:** Each individual digit proportion ≈ Benford expected  
- **Reject H₀ for digit d** if $|Z_d| > 1.96$ (two-tailed, $\alpha = 0.05$)

In [6]:
def zscore_test(leading_digits: pd.Series) -> dict:
    n = len(leading_digits)
    observed_props = observed_frequencies(leading_digits) / n

    z_scores = []
    for i in range(len(DIGITS)):
        p_e = BENFORD_PROBS[i]
        p_o = observed_props[i]
        numerator = abs(p_o - p_e) - (1 / (2 * n))
        denominator = np.sqrt(p_e * (1 - p_e) / n)
        z_scores.append(numerator / denominator)

    z_scores = np.array(z_scores)
    significant_digits = DIGITS[np.abs(z_scores) > 1.96]

    return {
        "test": "Z-Score (per digit)",
        "z_scores": z_scores,
        "significant_digits": significant_digits,
        "reject_H0": len(significant_digits) > 0,
        "critical_value": 1.96,
        "n": n,
    }

### Test 5: Mantissa Arc Test

$$a = \frac{2}{n} \sqrt{\left(\sum_{i=1}^{n} \cos(2\pi m_i)\right)^2 + \left(\sum_{i=1}^{n} \sin(2\pi m_i)\right)^2}$$

- Maps mantissas onto the unit circle. If data is Benford-conforming, the mean vector should be approximately **(0, 0)**  
- **H₀:** Mean vector ≈ (0, 0)  
- **Reject H₀** if $a > 1.0$ (Morrow 2014 threshold)

In [7]:
def mantissa_arc_test(series: pd.Series) -> dict:
    mantissas = extract_mantissas(series)
    n = len(mantissas)
    angles = 2 * np.pi * mantissas

    cos_sum = np.sum(np.cos(angles))
    sin_sum = np.sum(np.sin(angles))
    arc_stat = (2 / n) * np.sqrt(cos_sum**2 + sin_sum**2)

    return {
        "test": "Mantissa Arc Test",
        "statistic": arc_stat,
        "critical_value": 1.0,
        "reject_H0": arc_stat > 1.0,
        "n": n,
        "mantissas": mantissas,
    }

## 4. Orchestration & Output

In [8]:
def run_all_tests(series: pd.Series, column_name: str) -> dict:
    leading_digits = extract_leading_digits(series)
    return {
        "column": column_name,
        "chi2": chi_squared_test(leading_digits),
        "ks": ks_test(series),
        "mad": mad_test(leading_digits),
        "zscore": zscore_test(leading_digits),
        "arc": mantissa_arc_test(series),
        "leading_digits": leading_digits,
    }


def print_results(results: dict):
    col = results["column"]
    sep = "=" * 60
    print(f"\n{sep}")
    print(f"  COLUMN: {col.upper()}")
    print(sep)

    r = results["chi2"]
    verdict = "REJECT H0" if r["reject_H0"] else "FAIL TO REJECT H0"
    print(f"\n[1] Chi-Squared Test          -> {verdict}")
    print(f"    X2 = {r['statistic']:.4f}  |  critical = {r['critical_value']:.4f}  |  p = {r['p_value']:.4e}")

    r = results["ks"]
    verdict = "REJECT H0" if r["reject_H0"] else "FAIL TO REJECT H0"
    print(f"\n[2] Kolmogorov-Smirnov Test   -> {verdict}")
    print(f"    D = {r['statistic']:.4f}  |  critical ~ {r['critical_value']:.4f}  |  p = {r['p_value']:.4e}")

    r = results["mad"]
    verdict = "REJECT H0" if r["reject_H0"] else "FAIL TO REJECT H0"
    print(f"\n[3] Mean Absolute Deviation   -> {verdict}")
    print(f"    MAD = {r['statistic']:.6f}  |  Conformity: {r['conformity']}")

    r = results["zscore"]
    verdict = "REJECT H0" if r["reject_H0"] else "FAIL TO REJECT H0"
    print(f"\n[4] Z-Score (per digit)       -> {verdict}")
    for i, d in enumerate(DIGITS):
        flag = " *" if abs(r["z_scores"][i]) > 1.96 else ""
        print(f"    Digit {d}: Z = {r['z_scores'][i]:6.3f}{flag}")
    if len(r["significant_digits"]) > 0:
        print(f"    Significant digits: {list(r['significant_digits'])}")

    r = results["arc"]
    verdict = "REJECT H0" if r["reject_H0"] else "FAIL TO REJECT H0"
    print(f"\n[5] Mantissa Arc Test         -> {verdict}")
    print(f"    Arc stat = {r['statistic']:.4f}  |  threshold = {r['critical_value']:.1f}")
    print(f"\n{sep}")


def print_summary_table(all_results: list):
    print(f"\n{'='*75}")
    print("  SUMMARY TABLE -- REJECT H0 (Yes = anomaly detected)")
    print(f"{'='*75}")
    print(f"{'Column':<20} {'Chi2':<12} {'KS':<12} {'MAD':<12} {'Z-Score':<12} {'Arc':<12}")
    print("-" * 75)
    for r in all_results:
        print(
            f"{r['column']:<20}"
            f"{'Yes' if r['chi2']['reject_H0'] else 'No':<12}"
            f"{'Yes' if r['ks']['reject_H0'] else 'No':<12}"
            f"{'Yes' if r['mad']['reject_H0'] else 'No':<12}"
            f"{'Yes' if r['zscore']['reject_H0'] else 'No':<12}"
            f"{'Yes' if r['arc']['reject_H0'] else 'No':<12}"
        )
    print(f"{'='*75}")

## 5. Load Dataset

In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wordsforthewise/lending-club")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'lending-club' dataset.
Path to dataset files: /kaggle/input/lending-club


In [15]:
df = pd.read_csv('/kaggle/input/lending-club/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv', low_memory=False)

print(f"Shape: {df.shape}")
df[["loan_amnt", "annual_inc", "funded_amnt"]].describe()

Shape: (2260701, 151)


,loan_amnt,annual_inc,funded_amnt
count,2.260668e+06,2.260664e+06,2.260668e+06
mean,1.504693e+04,7.799243e+04,1.504166e+04
std,9.190245e+03,1.126962e+05,9.188413e+03
min,5.000000e+02,0.000000e+00,5.000000e+02
25%,8.000000e+03,4.600000e+04,8.000000e+03
50%,1.290000e+04,6.500000e+04,1.287500e+04
75%,2.000000e+04,9.300000e+04,2.000000e+04
max,4.000000e+04,1.100000e+08,4.000000e+04


## 6. Run All Tests

In [16]:
columns = ["loan_amnt", "annual_inc", "funded_amnt"]
columns = [c for c in columns if c in df.columns]

all_results = []
for col in columns:
    series = pd.to_numeric(df[col], errors="coerce")
    results = run_all_tests(series, col)
    print_results(results)
    all_results.append(results)

print_summary_table(all_results)


  COLUMN: LOAN_AMNT

[1] Chi-Squared Test          -> REJECT H0
    X2 = 214987.6267  |  critical = 15.5073  |  p = 0.0000e+00

[2] Kolmogorov-Smirnov Test   -> REJECT H0
    D = 0.1803  |  critical ~ 0.0009  |  p = 0.0000e+00

[3] Mean Absolute Deviation   -> REJECT H0
    MAD = 0.031000  |  Conformity: Nonconformity

[4] Z-Score (per digit)       -> REJECT H0
    Digit 1: Z = 335.866 *
    Digit 2: Z = 146.181 *
    Digit 3: Z = 34.464 *
    Digit 4: Z = 254.463 *
    Digit 5: Z = 146.594 *
    Digit 6: Z = 104.305 *
    Digit 7: Z = 128.478 *
    Digit 8: Z = 11.947 *
    Digit 9: Z = 118.464 *
    Significant digits: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

[5] Mantissa Arc Test         -> FAIL TO REJECT H0
    Arc stat = 0.3945  |  threshold = 1.0


  COLUMN: ANNUAL_INC

[1] Chi-Squared Test          -> REJECT H0
    X2 = 583749.2371  |  critical = 15.5073  |  p = 0.0000e+00

[2] Kolmogorov-Smirnov Test